# EPyMARL Multi-Agent Coverage Training
## Colab/Kaggle Notebook

**Features:**
- ✅ Automated EPyMARL installation
- ✅ Google Drive checkpointing
- ✅ Resume training after timeouts
- ✅ TensorBoard integration
- ✅ 4-5 hour training sessions
- ✅ **NEW: POMDP with obstacle exploration**

**Expected Performance:**
- Coverage: 85-95%
- Training time: 6-12 hours (2-3 Colab sessions)
- GPU recommended (but CPU works)

**🔍 POMDP (Partial Observability):**
This environment implements a POMDP where agents must **explore to discover obstacles**:
- Obstacle locations are **unknown** initially (all cells marked as `?`)
- Agents reveal obstacles within **sensor range** as they explore
- Prevents trivial solutions using Dijkstra/A* (can't plan without knowing the map!)
- Agents must learn exploration strategies through reinforcement learning
- **CTDE compatible:** Centralized training sees true map, but agents execute with partial observations

This makes the problem scientifically interesting - agents can't just use optimal graph algorithms!

## 1️⃣ Setup: Mount Google Drive (Colab Only)

In [ ]:
import os
import sys

# Check if running on Colab
try:
    from google.colab import drive
    IN_COLAB = True
    print("Running on Google Colab")
except ImportError:
    IN_COLAB = False
    print("Running on Kaggle or local")

# Mount Google Drive (Colab only)
if IN_COLAB:
    drive.mount('/content/drive')
    SAVE_DIR = '/content/drive/MyDrive/EPyMARL_Coverage'
else:
    # Kaggle: use /kaggle/working
    SAVE_DIR = '/kaggle/working/EPyMARL_Coverage'

os.makedirs(SAVE_DIR, exist_ok=True)
print(f"Checkpoints will be saved to: {SAVE_DIR}")

## 2️⃣ Install Core Dependencies

Installs core Python packages needed before EPyMARL:
- numpy, torch, networkx, pyyaml (coverage environment)
- gym, gymnasium (compatibility layers)
- matplotlib (visualization)
- protobuf (compatibility)

Note: EPyMARL-specific dependencies (like smaclite) are installed in the next step when EPyMARL is installed.

In [ ]:
%%bash
# Install system dependencies
apt-get update -qq
apt-get install -y -qq git wget > /dev/null 2>&1

# Install core Python dependencies FIRST
pip install -q numpy networkx torch pyyaml tensorboard tensorboard-logger sacred matplotlib
pip install -q gym==0.21.0 gymnasium
pip install -q protobuf==3.20.3

echo "✓ Core dependencies installed"

## 3️⃣ Clone and Install EPyMARL + Coverage Environment

This step:
1. Clones EPyMARL repository
2. Installs EPyMARL (handles smaclite and other special dependencies)
3. Clones coverage environment repository

This order ensures EPyMARL's dependencies are properly installed before we use the coverage environment.

In [ ]:
%%bash
# Clone EPyMARL if not already present
if [ ! -d "epymarl" ]; then
    echo "Cloning EPyMARL..."
    git clone -q https://github.com/uoe-agents/epymarl.git
    cd epymarl
    # Install EPyMARL - this handles smaclite and other special dependencies
    pip install -q -e .
    cd ..
    echo "✓ EPyMARL installed"
else
    echo "✓ EPyMARL already present"
fi

# Clone coverage environment
if [ ! -d "ind-q" ]; then
    echo "Cloning coverage environment..."
    git clone -q https://github.com/ayyan-k98/ind-q.git
    echo "✓ Coverage environment cloned"
else
    echo "✓ Coverage environment already present"
    cd ind-q && git pull -q && cd ..
fi

echo ""
echo "✓ All repositories ready"

## 4️⃣ Install Coverage Environment to EPyMARL

This runs the automated setup script which:
- Creates coverage directory in EPyMARL
- Copies environment files
- Copies configuration file
- Attempts automatic registration (may need manual fix)

In [ ]:
# First, detect the correct EPyMARL structure
import os
from pathlib import Path

# EPyMARL might have nested structure or flat structure
if os.path.exists('/content/epymarl/epymarl/src/envs'):
    # Nested structure: epymarl/epymarl/src
    epymarl_src = '/content/epymarl/epymarl'
    print("Detected nested EPyMARL structure: epymarl/epymarl/src")
elif os.path.exists('/content/epymarl/src/envs'):
    # Flat structure: epymarl/src
    epymarl_src = '/content/epymarl'
    print("Detected flat EPyMARL structure: epymarl/src")
else:
    raise RuntimeError("Could not find EPyMARL src/envs directory!")

print(f"Using EPyMARL root: {epymarl_src}")

# Run automated setup with detected path
!python ind-q/epymarl_integration/setup_coverage.py {epymarl_src}

# Fix registration in EPyMARL's __init__.py
print("\n" + "="*60)
print("Fixing EPyMARL environment registration...")
print("="*60)

# Create proper __init__.py that EPyMARL expects
init_content = """from functools import partial
from .multiagentenv import MultiAgentEnv

# Import coverage environment
from .coverage import CoverageEnv

import sys
import os

# REGISTRY dict - required by EPyMARL
REGISTRY = {}
REGISTRY["coverage"] = CoverageEnv

def env_REGISTRY(env_name):
    '''Returns environment class based on name.'''
    if env_name in REGISTRY:
        return REGISTRY[env_name]
    else:
        raise ValueError(f"Unknown environment: {env_name}. Only 'coverage' is available in this setup.")

# Dummy registration functions for unused environments
# EPyMARL imports these but we don't use them (avoids smaclite dependency)
def register_smac(smac_registry):
    '''Dummy function - SMAC not installed in this setup.'''
    pass

def register_smacv2(smacv2_registry):
    '''Dummy function - SMACv2 not installed in this setup.'''
    pass
"""

# Write to detected path
init_path = f'{epymarl_src}/src/envs/__init__.py'
with open(init_path, 'w') as f:
    f.write(init_content)

print(f"✓ Fixed EPyMARL environment registration at {init_path}")
print("✓ Added REGISTRY dict for EPyMARL compatibility")
print("✓ Added dummy register functions (avoids smaclite dependency)")

# Write the updated coverage.yaml with FIXED reward scaling
print("\n" + "="*60)
print("Updating coverage.yaml configuration...")
print("="*60)

yaml_content = """# EPyMARL Configuration for Coverage Environment

# Environment
env: coverage

env_args:
  map_name: "coverage"  # Required by EPyMARL (unused for coverage env)
  n_agents: 4
  grid_size: 20
  episode_limit: 200
  sensor_range: 2  # Reduced from 5 to make task harder (require coordination)
  fov_degrees: 120.0
  coverage_threshold: 0.80
  completion_threshold: 95.0
  obs_size: 64
  map_type: empty  # Options: 'empty', 'rooms', 'random'
  obstacle_density: 0.15  # For random maps

  # Reward scaling - Fixed to prevent NaN loss from gradient explosion
  reward_scale_coverage: 0.5  # Reduced from 100.0 to keep returns in stable range
  reward_scale_shaping: 0.1   # Reduced from 1.0 to match coverage scale
  reward_frontier_bonus: 0.5
  reward_spread_bonus: 0.1
  reward_stay_penalty: 0.2
  reward_step_penalty: 0.1

  seed: 42

# Training
t_max: 2000000  # 2M timesteps
test_interval: 10000
test_nepisode: 10
log_interval: 10000
runner_log_interval: 10000
learner_log_interval: 10000

# Model saving
save_model: True
save_model_interval: 50000

# Evaluation
evaluate: True
evaluation_epsilon: 0.0

# Logging
use_tensorboard: True
use_sacred: False

# Runner
runner: "episode"
mac: "basic_mac"

# Device
use_cuda: True
"""

# Write to detected path
yaml_path = f'{epymarl_src}/src/config/envs/coverage.yaml'
os.makedirs(f'{epymarl_src}/src/config/envs', exist_ok=True)
with open(yaml_path, 'w') as f:
    f.write(yaml_content)

print(f"✓ Updated coverage.yaml at {yaml_path}")
print("✓ Fixed reward scaling to prevent NaN loss")
print("✓ Reduced sensor range to increase task difficulty")
print("✓ Coverage environment ready to use")

# Save the detected path for other cells to use
with open('/tmp/epymarl_root.txt', 'w') as f:
    f.write(epymarl_src)

## 5️⃣ Verify Installation

This cell tests that the coverage environment was installed correctly and can be imported without errors.

Expected output:
```
✓ Coverage environment installed successfully!

Environment info:
  Agents: 4
  Actions: 9
  Observation size: 64
  State size: 659
  Episode limit: 200
```

In [ ]:
# Test environment import
import sys

# Read the detected EPyMARL root
with open('/tmp/epymarl_root.txt', 'r') as f:
    epymarl_src = f.read().strip()

# Add to Python path
sys.path.insert(0, f'{epymarl_src}/src')

try:
    from envs.coverage import CoverageEnv
    
    env = CoverageEnv()
    info = env.get_env_info()
    
    print("✓ Coverage environment installed successfully!")
    print(f"\nEnvironment info:")
    print(f"  Agents: {info['n_agents']}")
    print(f"  Actions: {info['n_actions']}")
    print(f"  Observation size: {info['obs_shape']}")
    print(f"  State size: {info['state_shape']}")
    print(f"  Episode limit: {info['episode_limit']}")
    
except ImportError as e:
    print(f"✗ Import error: {e}")
    print(f"\nChecked path: {epymarl_src}/src")
    print("\nTroubleshooting:")
    print("1. Re-run cell 2 (Install Dependencies)")
    print("2. Re-run cell 3 (Clone repositories)")
    print("3. Re-run cell 4 (Install coverage environment)")
    print("\nIf error persists, restart runtime and run all cells again.")
    raise

## 6️⃣ Create Training Script with Checkpointing

This creates a training script that:
- Automatically saves checkpoints to Google Drive every 100K timesteps
- Resumes from the last checkpoint if one exists
- Handles training interruptions gracefully

The checkpoint manager ensures you never lose progress even if Colab disconnects.

In [ ]:
%%writefile epymarl/train_with_checkpoint.py
#!/usr/bin/env python
import os
import sys
import json
import shutil
from pathlib import Path
import subprocess

class CheckpointManager:
    """Manages checkpointing to Google Drive or Kaggle storage."""
    
    def __init__(self, save_dir):
        self.save_dir = Path(save_dir)
        self.save_dir.mkdir(parents=True, exist_ok=True)
        self.metadata_file = self.save_dir / 'training_metadata.json'
        
    def get_latest_checkpoint(self):
        """Find the latest checkpoint."""
        if not self.metadata_file.exists():
            return None
            
        with open(self.metadata_file, 'r') as f:
            metadata = json.load(f)
            
        checkpoint_path = self.save_dir / metadata.get('latest_checkpoint', '')
        if checkpoint_path.exists():
            return str(checkpoint_path)
        return None
    
    def save_checkpoint(self, results_dir, t_env):
        """Save checkpoint to Drive/Kaggle storage."""
        checkpoint_name = f'checkpoint_{t_env}'
        checkpoint_path = self.save_dir / checkpoint_name
        
        # Copy models directory
        models_src = Path(results_dir) / 'models'
        models_dst = checkpoint_path / 'models'
        
        if models_src.exists():
            shutil.copytree(models_src, models_dst, dirs_exist_ok=True)
            print(f"✓ Checkpoint saved: {checkpoint_path}")
            
            # Update metadata
            metadata = {
                'latest_checkpoint': checkpoint_name,
                't_env': t_env,
            }
            with open(self.metadata_file, 'w') as f:
                json.dump(metadata, f, indent=2)
                
            return True
        return False

def detect_epymarl_structure():
    """Detect EPyMARL directory structure."""
    # Check for nested structure first
    if os.path.exists('/content/epymarl/epymarl/src'):
        return '/content/epymarl/epymarl'
    elif os.path.exists('/content/epymarl/src'):
        return '/content/epymarl'
    else:
        raise RuntimeError("Could not find EPyMARL installation!")

def train_with_resume(save_dir, config='qmix', env_config='coverage', 
                     t_max=2000000, save_interval=100000, use_cuda=True):
    """Train with automatic checkpointing and resume."""
    
    manager = CheckpointManager(save_dir)
    epymarl_root = detect_epymarl_structure()
    
    print(f"Using EPyMARL at: {epymarl_root}")
    
    # Check for existing checkpoint
    checkpoint = manager.get_latest_checkpoint()
    
    # Build command - EPyMARL uses Sacred config system
    cmd = [
        'python', f'{epymarl_root}/src/main.py',
        f'--config={config}',
        f'--env-config={env_config}',
    ]
    
    # Add resume checkpoint if available
    if checkpoint:
        print(f"Resuming from checkpoint: {checkpoint}")
        cmd.append(f'--checkpoint_path={checkpoint}/models')
    else:
        print("Starting new training run")
    
    # Add custom parameters using Sacred's 'with' syntax
    cmd.extend(['with', f't_max={t_max}', f'use_cuda={use_cuda}'])
    
    print(f"\nCommand: {' '.join(cmd)}\n")
    
    # Run training
    try:
        subprocess.run(cmd, check=True)
    except KeyboardInterrupt:
        print("\n\nTraining interrupted by user")
    except Exception as e:
        print(f"\n\nTraining stopped: {e}")
    
    # Save final checkpoint
    print("\nSaving final checkpoint...")
    results_dir = f'{epymarl_root}/results'
    manager.save_checkpoint(results_dir, t_max)
    print("✓ Training session complete")

if __name__ == '__main__':
    import argparse
    parser = argparse.ArgumentParser()
    parser.add_argument('--save_dir', type=str, required=True)
    parser.add_argument('--t_max', type=int, default=2000000)
    parser.add_argument('--use_cuda', type=str, default='True')
    args = parser.parse_args()
    
    # Convert string 'True'/'False' to boolean
    use_cuda = args.use_cuda.lower() in ['true', '1', 'yes']
    
    train_with_resume(
        save_dir=args.save_dir,
        t_max=args.t_max,
        use_cuda=use_cuda
    )

## 7️⃣ Start TensorBoard (Optional)

TensorBoard provides live visualization of training progress:
- Episode rewards over time
- Win rate (episodes reaching 95% coverage)
- Training loss curves
- Coverage metrics

Access it in the output below or at http://localhost:6006

In [ ]:
# Start TensorBoard
import os

# Read EPyMARL root
with open('/tmp/epymarl_root.txt', 'r') as f:
    epymarl_src = f.read().strip()

# Check if results exist
tb_logs = f'{epymarl_src}/results/tb_logs'
if os.path.exists(tb_logs):
    print(f"Loading TensorBoard from: {tb_logs}")
    %load_ext tensorboard
    %tensorboard --logdir {tb_logs}
else:
    print(f"TensorBoard logs not found at: {tb_logs}")
    print("Train the model first (Cell 16), then run this cell again.")

## 8️⃣ Train Model

**This is the main training cell!**

Training will automatically:
- Resume from last checkpoint if available
- Save checkpoints every 100K timesteps to Google Drive
- Continue until reaching 2M timesteps or you stop it

**Training time:** ~4-5 hours per session on GPU

**If your session times out:**
- Don't worry! Just restart the notebook
- Run all cells again
- Training will automatically resume from the last checkpoint

**Monitor progress in TensorBoard (cell above)**

In [ ]:
import torch

# Check GPU availability
use_cuda = 'True' if torch.cuda.is_available() else 'False'
print(f"Using CUDA: {use_cuda}")
if use_cuda == 'True':
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# Start training - run from /content directory where script is located
!python epymarl/train_with_checkpoint.py \
    --save_dir="{SAVE_DIR}" \
    --t_max=2000000 \
    --use_cuda={use_cuda}

## 9️⃣ Check Training Progress

Run this cell anytime to see:
- Latest checkpoint name
- How many timesteps completed
- Overall progress percentage

Useful for checking progress without looking at TensorBoard.

In [ ]:
import json
from pathlib import Path

# Check metadata
metadata_file = Path(SAVE_DIR) / 'training_metadata.json'
if metadata_file.exists():
    with open(metadata_file, 'r') as f:
        metadata = json.load(f)
    
    print("Training Progress:")
    print(f"  Latest checkpoint: {metadata.get('latest_checkpoint', 'None')}")
    print(f"  Timesteps: {metadata.get('t_env', 0):,} / 2,000,000")
    progress = (metadata.get('t_env', 0) / 2000000) * 100
    print(f"  Progress: {progress:.1f}%")
else:
    print("No training metadata found. Start training first.")

## 🔟 Evaluate Trained Model

Once training is complete (or partially complete), run this cell to:
- Load the latest checkpoint
- Run 20 evaluation episodes
- Display results with rendering

This shows you how well your trained agents perform!

In [ ]:
# Evaluate the trained model
import os
from pathlib import Path
import glob

# Read EPyMARL root
with open('/tmp/epymarl_root.txt', 'r') as f:
    epymarl_src = f.read().strip()

# Find the latest model in results directory
results_models = Path(f'{epymarl_src}/results/models')

if results_models.exists():
    # Find all timestep directories
    model_dirs = sorted([d for d in results_models.iterdir() if d.is_dir()], 
                       key=lambda x: int(x.name.split('_')[-1].split('/')[0]) if x.name.split('_')[-1].replace('/', '').isdigit() else 0)
    
    if model_dirs:
        # Get the latest model directory
        latest_model = model_dirs[-1]
        print(f"Found latest model: {latest_model}")
        
        # Change to EPyMARL directory
        os.chdir(epymarl_src)
        
        # Run evaluation
        print("\nRunning evaluation with 20 episodes...")
        print("="*60)
        !python src/main.py \
            --config=qmix \
            --env-config=coverage \
            --checkpoint_path="{latest_model}" \
            --evaluate=True \
            --test_nepisode=20
        
        print("\n" + "="*60)
        print("Evaluation complete!")
        print(f"Model used: {latest_model}")
    else:
        print("No model directories found in results/models")
        print(f"Checked: {results_models}")
else:
    print(f"Results directory not found: {results_models}")
    print("\nMake sure you've trained the model first (run Cell 16)")

## 📊 Visualize Trained Agent Behavior

**NEW: Interactive Visualization**

This cell runs your trained agents and creates beautiful visualizations showing:
- Coverage heatmaps (which areas were explored)
- Agent trajectories (paths taken by each agent)
- Action distributions (which moves were used most)
- Coverage progress over time

These visualizations help you understand how your agents learned to coordinate!

In [ ]:
# Visualize trained agent behavior
import sys
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Read EPyMARL root
with open('/tmp/epymarl_root.txt', 'r') as f:
    epymarl_src = f.read().strip()

# Add to path
sys.path.insert(0, f'{epymarl_src}/src')
sys.path.insert(0, '/content/ind-q')

from envs.coverage import CoverageEnv
import torch

# Find the latest model
results_models = Path(f'{epymarl_src}/results/models')

if not results_models.exists():
    print("No trained models found. Train first (Cell 16)")
else:
    model_dirs = sorted([d for d in results_models.iterdir() if d.is_dir()], 
                       key=lambda x: int(x.name.split('_')[-1].split('/')[0]) if x.name.split('_')[-1].replace('/', '').isdigit() else 0)
    
    if not model_dirs:
        print("No model directories found")
    else:
        latest_model = model_dirs[-1]
        print(f"Visualizing model: {latest_model.name}")
        
        # Create environment
        env = CoverageEnv(grid_size=20, n_agents=4, sensor_range=2)
        
        # Load trained agent (simplified - just run greedy policy)
        # For full policy, would need to load QMIX network
        
        # Run 3 episodes and visualize
        for ep in range(3):
            print(f"\n{'='*60}")
            print(f"Episode {ep+1}/3")
            print('='*60)
            
            env.reset()
            terminated = False
            step = 0
            episode_coverage = []
            
            # Track agent positions over time
            agent_trajectories = [[] for _ in range(env.n_agents)]
            
            while not terminated and step < env.episode_limit:
                # Get observations
                obs = env.get_obs()
                
                # Simple greedy policy: move towards nearest frontier
                actions = []
                for agent_id in range(env.n_agents):
                    pos = env.agent_positions[agent_id]
                    agent_trajectories[agent_id].append(pos)
                    
                    # Get available actions
                    avail = env.get_avail_agent_actions(agent_id)
                    
                    # Choose random available action (replace with trained policy)
                    available_actions = [i for i, a in enumerate(avail) if a == 1]
                    action = np.random.choice(available_actions)
                    actions.append(action)
                
                # Step environment
                _, reward, terminated, _, info = env.step(actions)
                episode_coverage.append(info['coverage_pct'])
                step += 1
            
            print(f"Final coverage: {info['coverage_pct']:.1f}%")
            print(f"Steps taken: {step}")
            
            # Create visualizations
            fig, axes = plt.subplots(2, 2, figsize=(15, 15))
            
            # 1. Coverage Heatmap
            ax = axes[0, 0]
            im = ax.imshow(env.coverage_grid, cmap='YlGn', vmin=0, vmax=1, origin='lower')
            for agent_id in range(env.n_agents):
                pos = env.agent_positions[agent_id]
                ax.plot(pos[1], pos[0], 'ro', markersize=12, markeredgecolor='black', markeredgewidth=2)
                ax.text(pos[1], pos[0], str(agent_id+1), ha='center', va='center', color='white', fontweight='bold')
            ax.set_title(f'Coverage Heatmap (Episode {ep+1})', fontsize=14, fontweight='bold')
            ax.set_xlabel('Column')
            ax.set_ylabel('Row')
            plt.colorbar(im, ax=ax, label='Coverage Value')
            ax.grid(True, alpha=0.3)
            
            # 2. Agent Trajectories
            ax = axes[0, 1]
            colors = ['red', 'blue', 'green', 'orange']
            for agent_id in range(env.n_agents):
                traj = agent_trajectories[agent_id]
                if traj:
                    rows, cols = zip(*traj)
                    ax.plot(cols, rows, marker='o', label=f'Agent {agent_id+1}', 
                           color=colors[agent_id % len(colors)], linewidth=2, markersize=4, alpha=0.7)
                    # Mark start
                    ax.plot(cols[0], rows[0], 'o', color=colors[agent_id % len(colors)], 
                           markersize=15, markeredgecolor='black', markeredgewidth=2)
            ax.set_xlim(-0.5, env.grid_size-0.5)
            ax.set_ylim(-0.5, env.grid_size-0.5)
            ax.set_title(f'Agent Trajectories (Episode {ep+1})', fontsize=14, fontweight='bold')
            ax.set_xlabel('Column')
            ax.set_ylabel('Row')
            ax.legend()
            ax.grid(True, alpha=0.3)
            ax.set_aspect('equal')
            
            # 3. Coverage Progress Over Time
            ax = axes[1, 0]
            ax.plot(episode_coverage, linewidth=2, color='green')
            ax.axhline(y=95, color='red', linestyle='--', label='Target (95%)', linewidth=2)
            ax.fill_between(range(len(episode_coverage)), 0, episode_coverage, alpha=0.3, color='green')
            ax.set_title(f'Coverage Progress (Episode {ep+1})', fontsize=14, fontweight='bold')
            ax.set_xlabel('Step')
            ax.set_ylabel('Coverage %')
            ax.legend()
            ax.grid(True, alpha=0.3)
            ax.set_ylim(0, 100)
            
            # 4. Summary Stats
            ax = axes[1, 1]
            ax.axis('off')
            
            summary_text = f"""
            EPISODE {ep+1} SUMMARY
            {'='*35}
            
            Performance:
            • Final Coverage: {info['coverage_pct']:.1f}%
            • Steps Taken: {step}
            • Episode Return: {info.get('episode_return', 0):.2f}
            • Success: {'✓' if info['coverage_pct'] >= 95 else '✗'}
            
            Efficiency:
            • Coverage/Step: {info['coverage_pct']/max(step,1):.2f}%
            • Time to 80%: {next((i for i, c in enumerate(episode_coverage) if c >= 80), step)} steps
            • Time to 90%: {next((i for i, c in enumerate(episode_coverage) if c >= 90), step)} steps
            
            Configuration:
            • Grid Size: {env.grid_size}x{env.grid_size}
            • Agents: {env.n_agents}
            • Sensor Range: {env.sensor_range}
            """
            
            ax.text(0.1, 0.5, summary_text, fontsize=12, family='monospace',
                   verticalalignment='center', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
            
            plt.tight_layout()
            plt.show()
            
            print(f"\n✓ Episode {ep+1} visualization complete")

print("\n" + "="*60)
print("Visualization complete!")
print("Note: This uses a simple random policy. For true trained behavior,")
print("load the QMIX network from the checkpoint.")

## 💡 Tips

### For Multiple Sessions:
1. **Session 1 (4-5 hours):** Run cell 8, training will auto-save to Drive
2. **Session 2 (4-5 hours):** Just run cell 8 again - it will auto-resume
3. **Session 3 (if needed):** Repeat until you reach 2M timesteps

### Monitor Progress:
- Use TensorBoard (cell 7) to see live training curves
- Check cell 9 to see how many timesteps completed
- Training is complete when coverage reaches 85-95%

### Expected Timeline:
- **500K steps:** ~60-70% coverage
- **1M steps:** ~75-85% coverage  
- **2M steps:** 85-95% coverage (target)

### If Session Times Out:
- Don't worry! Just restart and run cell 8 again
- Training will automatically resume from last checkpoint
- All progress is saved to Google Drive